# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in two libraries, one `%pip install`, and zero
credentials. A geojson polygon becomes a morton MOC; the MOC checks itself
against the store's own coverage; the covered shards open with timings; one
shard renders in 3-D (ATL03 + GEDI together); the current view exports to
voxel cubes on any grid you name and saves to disk.

Everything below is reader-side and calls no zagg public API — `mortie` for
the geometry, `moczarr` for the store, plus the t-digest algebra that
`moczarr[zagg]` imports from zagg rather than vendoring (moczarr issue #19).
That algebra is the only zagg code on the path. It all runs anonymously
against public S3, binder-ready.

Its sibling is [`waveform_viewer.ipynb`](waveform_viewer.ipynb), which takes
the same polygon and stores down to the cell-level join: one GEDI o18
footprint against the 2×2 ATL03 o19 cells beneath it, both rebuilt from their
stored t-digests. The two are separate notebooks because this one needs
`%matplotlib widget` and that one `%matplotlib inline`; the backends collide
in a single kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets
%matplotlib widget

import os
import time
import zipfile
from io import BytesIO

import moczarr as mz
import numpy as np
from moczarr.hhdc import (
    block_rank,
    chunk_z_range,
    rank_to_rowcol,
    rasterize_cell,
    rowcol_to_rank,
)
from mortie import generate_morton_children, moc

# The drawing lives in viewers.py beside this notebook, so the cells below stay
# about the READ path. Both demo notebooks share it.
from viewers import BLOCK_ORDER, SIDE, UNITS, block_of, human_bytes, view3d

# One store per product, each appendable. Coverage answers which ground the
# store holds; the store name never does.
STORES = {
    "atl03": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
        "19/h_tdigest_signal",
    ),
    "gedi": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr",
        "18/rx_flux",
    ),
}
S3 = {"region": "us-west-2", "anonymous": True}

## One polygon in, covered shards out

Replace the polygon with any area within California or a NEON AOP site; the
cell tests whether each store actually covers it. An AOI in Alaska will pass
the ATL03 check and fail the GEDI one — GEDI flies on the ISS, so it sees no
higher than |lat| 51.6 and the Alaska NEON sites are outside its reach.

In [ ]:
aoi = {
    "features": [
        {
            "geometry": {
                "coordinates": [
                    [  # a ~4 km box on the SERC tract
                        [-76.56, 38.87],
                        [-76.50, 38.87],
                        [-76.50, 38.91],
                        [-76.56, 38.91],
                        [-76.56, 38.87],
                    ]
                ]
            }
        }
    ]
}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not contain the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)

# Everything below works ONE shard, named here and only here. Change the index
# to look at another -- opening one shard and viewing another leaves every read
# asking for a subtree outside the open leaf's axis, which comes back as a
# warning and an empty pane rather than an error.
SHARD = shards[0]
print(f"{len(shards)} shards cover the polygon: {shards}\nworking {SHARD}")

## Open one shard — every dataset, timed

In [ ]:
def open_shard(shard):
    """Open each store's leaf for this shard, and price a sweep of ONE field.

    The sweep decodes every stored digest of the named field across all 64
    blocks of the shard -- so the seconds and megabytes below are the cost of
    reading one ARRAY end to end, not of reading the leaf. This ATL03 leaf holds
    nine arrays (46.7 MiB on S3): the signal digests read here, their `locations`
    and `times` companions, the whole `h_tdigest_noise` channel with companions
    of its own, plus `morton`, `composition` and `count`. Reading the leaf costs
    several times what this prints.

    Nor is it every photon: signal only. On this shard `19/count` sums to
    3,010,061, which is the 1,266,765 signal centroids plus 1,743,296 noise ones.

    None of what it decodes is kept -- the viewer below fetches one block at a
    time, which is what holds this notebook inside Binder's 2 GB when the sparse
    stored digests are cast to the dense tensors visualization needs. The
    per-block cell and observation counts fall out of the same pass for free, so
    they come back too and label the viewer's block dropdown. They are STORED
    totals -- the viewer's own per-read counts are clipped to a finite z window
    and run lower.
    """
    handles, tally = {}, {}
    for name, (root, field) in STORES.items():
        t0 = time.perf_counter()
        store = mz.open_leaf(root, shard, **S3)
        _, element = mz.open_ragged(store, field)
        per, cells, centroids, nbytes, obs = {}, 0, 0, 0, 0.0
        for word, value in mz.read_ragged(store, field):
            v = np.asarray(value)
            weight = float(v[:, 1].sum())  # column 1 is the centroid weight
            block = int(block_of(word, BLOCK_ORDER))
            had_cells, had_obs = per.get(block, (0, 0.0))
            per[block] = (had_cells + 1, had_obs + weight)
            cells, centroids, nbytes, obs = (
                cells + 1,
                centroids + len(v),
                nbytes + v.nbytes,
                obs + weight,
            )
        tally[name] = per
        print(
            f"{name:6s} swept {field} in {time.perf_counter() - t0:5.1f}s — "
            f"{cells:,} cells over {len(per)} blocks, {centroids:,} centroids, "
            f"{obs:,.0f} {UNITS[name]}, {nbytes / 2**20:.1f} MiB decoded "
            f"(this ONE array, not the whole leaf)"
        )
        print(f"       element {element}")
        handles[name] = (store, field)
    return handles, tally


handles, tally = open_shard(SHARD)


## The 3-D view — both sensors, exact centroids, time-aware

In [ ]:
view = view3d(handles, SHARD, tally=tally)  # blocks sorted densest-coincident first


## Export what you see — voxel cubes, on a grid you choose

Two exports. Both leave the `read_tensors` path and build the cube directly
from the stored digests, because both need something `read_tensors` cannot
give: an arbitrary output order for the first, and a *shared* z axis for the
second.

**1 — ATL03 alone, as fine as you like.** ATL03's digests carry a located
companion: one order-29 point word per centroid. That is what lets the cube be
finer than the o19 cells the digests are keyed by — truncate each point word to
the requested order and the photon lands in its own voxel. The default is
**o22, 1.554 m, and the z bin is set to match**, so the voxels are cubes and
not slabs. A whole o12 block at o22 is 1024² × 128, so the block is emitted as
`128³` chips instead — 8 × 8 of them, each a 199 m cube — and only the chips
that hold data are written.

Finer is one argument away and the arithmetic is the same, but read the fill
before you ask for it: ATL03's sampling is what it is, so a finer grid buys
detail along-track and empty voxels everywhere else.

| order | voxel | chip covers | xy fill in a 128-chip | full block, dense |
|---|---|---|---|---|
| o19 | 12.4 m | 1592 m | 9.2% | 0.01 GiB |
| **o22** | **1.554 m** | **199 m** | **4.1%** | 0.50 GiB |
| o24 | 0.389 m | 49.7 m | 0.7% | 8.00 GiB |

**2 — ATL03 and GEDI co-registered.** Two cubes of identical shape on one xy
lattice and one z axis, ready to stack. The xy grid is o19 — ATL03's own cell
order, kept rather than merged away, with each GEDI o18 cell **replicated into
its four o19 children**. That keeps ICESat-2's fine structure at the cost of
four copies of every GEDI observation, which is the trade the science team
asked for.

The z axis matters as much as the xy one and is easier to miss. `read_tensors`
derives its window per sensor: on this block it returns `z0 = -71.0` for ATL03
and `z0 = -59.0` for GEDI, so those two tensors are 6 bins out of register
before anyone stacks them. Here `chunk_z_range` is handed **both** sensors'
digests at once and returns one `(z0, dz)`, and every cell of both is
rasterized onto it.


In [ ]:
def voxel_chips(block, sensor="atl03", order=22, side=128, n_bins=128, path=None):
    """One o12 block -> isotropic `side`**3 count chips, empty chips skipped.

    Placement comes from the LOCATED companion, not the cell key: `block_rank`
    normalizes each centroid's order-29 point word and returns its block-local
    nested rank, and shifting that rank right by `2 * (29 - order)` is the same
    word truncated to `order` -- nested ranks are hierarchical, so no re-decode.
    That is the whole reason the cube can be finer than the o19 cells.

    The z bin is the cell edge, so a voxel is a cube. Each chip gets its own
    `z0` (the floor of its own minimum) rather than one window for the block:
    at `n_bins` bins a chip spans `side * dz` metres of z, and a block's full
    relief rarely fits in that, so a shared floor would spend most bins empty.
    `z0` is saved per chip, so absolute elevation is always recoverable.
    """
    store, field = handles[sensor]
    assert side and not side & (side - 1), "side must be a power of two"
    dz = SIDE / 2 ** (order - BLOCK_ORDER)  # ISOTROPIC: z bin == cell edge
    depth = order - (side.bit_length() - 1) - BLOCK_ORDER  # chip order below the block
    k = 2**depth  # chips across the block edge
    tiles = generate_morton_children(int(block), order - (side.bit_length() - 1))

    t0 = time.perf_counter()
    zs, wts, locs = [], [], []
    for row in mz.read_ragged(store, field, locations=True, subtree=mz.morton_decimal(int(block))):
        v = np.asarray(row[1])
        zs.append(v[:, 0])
        wts.append(v[:, 1])
        locs.append(np.asarray(row[2], dtype=np.uint64))
    z, wt = np.concatenate(zs), np.concatenate(wts)
    rank = block_rank(np.concatenate(locs), BLOCK_ORDER)[0] >> np.uint64(2 * (29 - order))
    row, col = rank_to_rowcol(rank, order - BLOCK_ORDER)
    read_s = time.perf_counter() - t0

    # Stream each chip into the archive and drop it: 64 chips x 8 MiB is 512 MiB
    # held at once, and mybinder.org caps the whole container at 2 GB. An `.npz`
    # is a zip of `.npy` members, so `np.load` reads this back unchanged.
    path = path or f"{sensor}_o{order}_chips_{mz.morton_decimal(int(block))}.npz"
    t1 = time.perf_counter()
    kept, filled, dropped, dense = {}, 0, 0, 0
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for i in range(k):
            for j in range(k):
                m = np.flatnonzero((row // side == i) & (col // side == j))
                if not len(m):
                    continue  # empty chip: not written at all
                z0 = np.floor(z[m].min())
                iz = ((z[m] - z0) / dz).astype(np.int64)
                dropped += int((iz >= n_bins).sum())  # relief past the window
                m, iz = m[iz < n_bins], iz[iz < n_bins]
                acc = np.zeros((side, side, n_bins), dtype=np.float32)
                np.add.at(acc, (row[m] % side, col[m] % side, iz), wt[m])
                chip = np.rint(acc).astype(np.uint32)  # a merged centroid's weight is fractional
                name = mz.morton_decimal(int(tiles[rowcol_to_rank(i, j, depth=depth)]))
                buf = BytesIO()
                np.save(buf, chip)
                zf.writestr(f"{name}.npy", buf.getvalue())
                kept[name] = {"z0": float(z0), "voxels": int(chip.astype(bool).sum())}
                filled += kept[name]["voxels"]
                dense += chip.nbytes
    write_s, on_disk = time.perf_counter() - t1, os.path.getsize(path)

    print(f"{sensor} o{order} chips — {dz:.3f} m isotropic voxels, {side}^3 = {dz * side:.0f} m cube")
    print(
        f"  read   {len(z):,} centroids, {wt.sum():,.0f} {UNITS[sensor]} in {read_s:.1f}s"
        + (f"  ({dropped:,} above the z window)" if dropped else "")
    )
    print(
        f"  wrote  {len(kept)} of {k * k} chips ({k * k - len(kept)} skipped empty), "
        f"{filled:,} voxels filled, in {write_s:.1f}s"
    )
    print(
        f"         {human_bytes(dense)} dense -> {human_bytes(on_disk)} in {path} "
        f"({dense / max(on_disk, 1):.0f}x deflated)"
    )
    return path, kept


In [ ]:
chips, manifest = voxel_chips(view.block)  # the block the viewer is on, at o22

# Finer is one argument away. At o24 the voxel is 0.389 m and the chip covers
# 49.7 m, so the same block becomes 32 x 32 = 1,024 chips instead of 64 -- far
# more of them, most skipped as empty, and a much slower write for the ones
# that are not. Uncomment to see both summaries side by side.
# chips24, _ = voxel_chips(view.block, order=24)


In [ ]:
def registered_pair(block, order=19, n_bins=128, resolution=0.5, path=None):
    """Both sensors as cubes of ONE shape, on one xy lattice and one z axis.

    xy: every cell is placed by the deinterleave of its block-local nested rank
    and then replicated over the `2**(order - cell_order)` children it covers.
    At `order=19` ATL03 is 1:1 and each GEDI o18 cell becomes 2x2 -- the
    science team's call, keeping ICESat-2's resolution rather than merging it
    up to GEDI's. Replication means a GEDI cube SUMS to four times its stored
    photoelectrons; the per-cell values are unchanged, and both are printed.

    z: `chunk_z_range` is handed both sensors' digests together, so the window
    it derives covers both and every cell is rasterized onto that one axis.
    Deriving it per sensor -- which is what `read_tensors` does -- puts the two
    12 m apart on this block, and nothing downstream would notice.
    """
    t0 = time.perf_counter()
    side = 2 ** (order - BLOCK_ORDER)
    per_sensor = {
        name: [
            (int(w), np.asarray(v))
            for w, v in mz.read_ragged(store, field, subtree=mz.morton_decimal(int(block)))
        ]
        for name, (store, field) in handles.items()
    }
    read_s = time.perf_counter() - t0

    z0, n_bins, dz = chunk_z_range(
        [d for got in per_sensor.values() for _, d in got],
        n_bins=n_bins,
        resolution=resolution,
        bottom=0.05,
        top=0.95,
        fit="degrade_resolution",
    )
    t1 = time.perf_counter()
    cubes, stats = {}, {}
    for name, (_store, field) in handles.items():
        cell_order = int(field.split("/", 1)[0])
        k = 2 ** (order - cell_order)  # children of one cell on the output grid
        cube = np.zeros((side, side, n_bins), dtype=np.float32)
        for w, digest in per_sensor[name]:
            rank = block_rank(np.uint64(w), BLOCK_ORDER)[0]
            r, c = rank_to_rowcol(rank, cell_order - BLOCK_ORDER)
            r, c = int(np.ravel(r)[0]) * k, int(np.ravel(c)[0]) * k
            cube[r : r + k, c : c + k, :] = rasterize_cell(digest, z0, dz, n_bins)
        cubes[name] = cube
        stats[name] = (cell_order, k, int((cube.sum(axis=2) > 0).sum()), float(cube.sum()))

    path = path or f"registered_o{order}_{mz.morton_decimal(int(block))}.npz"
    np.savez_compressed(path, **cubes, z0=z0, dz=dz, order=order)
    write_s, on_disk = time.perf_counter() - t1, os.path.getsize(path)
    dense = sum(c.nbytes for c in cubes.values())

    got = "as asked" if abs(dz - resolution) < 1e-9 else f"DEGRADED from the {resolution:g} m asked"
    print(f"registered pair o{order} — {next(iter(cubes.values())).shape} float32, both sensors")
    read = "; ".join(f"{n} {len(v):,} cells" for n, v in per_sensor.items())
    print(f"  read   {read} in {read_s:.1f}s")
    print(f"  grid   shared z = {z0:.1f} m + bin * {dz:g} m ({got})")
    for name, (co, k, cols, total) in stats.items():
        print(
            f"         {name}: {len(per_sensor[name]):,} o{co} cells -> {k}x{k} each -> "
            f"{cols:,} of {side * side:,} columns filled, {total:,.0f} {UNITS[name]}"
            + (f" ({total / k**2:,.0f} before {k**2}x replication)" if k > 1 else "")
        )
    print(f"  wrote  {human_bytes(dense)} dense -> {human_bytes(on_disk)} in {path}, {write_s:.1f}s")
    return path, cubes


In [ ]:
pair, cubes = registered_pair(view.block)

# They are registered, so they stack -- same shape, same lattice, same z axis.
stacked = np.stack([cubes["atl03"], cubes["gedi"]], axis=0)
print(f"stacked {stacked.shape} — {human_bytes(stacked.nbytes)}, ready for a 2-channel model")


Two libraries, one polygon — coverage, shards, timings, the
paired 3-D view, and co-registered voxel cubes on disk.
